# Import

In [1]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

c:\Users\jeons\anaconda3\envs\lg-hackathon\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Setting

In [2]:
MODEL_ID = "./base_model"     
OUT_DIR  = "./model"          

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512

# Quantization
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]

# Model Loads

In [6]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print("[INFO] 모델/토크나이저 로드 완료")

[INFO] 모델 로드 중...
[INFO] 모델/토크나이저 로드 완료


# Dataset Loads & Preprocess

In [4]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...


c:\Users\jeons\anaconda3\envs\lg-hackathon\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jeons\.cache\huggingface\hub\datasets--LGAI-EXAONE--MANTA-1M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regula

[INFO] 데이터 전처리 완료


# GPTQ Quantization

In [7]:
print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] GPTQ 완료")

[INFO] GPTQ 시작 (scheme=W4A16, samples=256, max_len=512)...


Tokenizing: 100%|██████████| 256/256 [00:00<00:00, 665.04 examples/s]

2026-02-04T19:42:37.405423+0900 | reset | INFO - Compression lifecycle reset
2026-02-04T19:42:37.406422+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-04T19:42:37.428536+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-04T19:42:37.429533+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-04T19:42:37.433728+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead



(1/31): Calibrating: 100%|██████████| 256/256 [02:04<00:00,  2.05it/s]

2026-02-04T19:44:42.209069+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 256 samples


...

(6/31): Calibrating: 100%|██████████| 256/256 [02:04<00:00,  2.05it/s]

2026-02-04T20:05:12.886089+0900 | compress_modules | INFO - Quantizing model.layers.5.self_attn.q_proj using 256 samples


2026-02-04T20:05:13.378415+0900 | compress | METRIC - time 0.49s
2026-02-04T20:05:13.379416+0900 | compress | METRIC - error 81.17
2026-02-04T20:05:13.398247+0900 | compress | METRIC - GPU 0 | usage: 10.56% | total memory: 12 GB
2026-02-04T20:05:13.398247+0900 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-04T20:05:13.402900+0900 | compress_modules | INFO - Quantizing model.layers.5.self_attn.k_proj using 256 samples
2026-02-04T20:05:13.712221+0900 | compress | METRIC - time 0.31s
2026-02-04T20:05:13.713218+0900 | compress | METRIC - error 23.84
2026-02-04T20:05:13.729339+0900 | compress | METRIC - GPU 0 | usage: 10.55% | total memory: 12 GB
2026-02-04T20:05:13.729339+0900 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-04T20:05:13.731430+0900 | compress_modules | INFO - Quantizing model.layers.5.self_attn.v_proj using 256 samples
2026-02-04T20:05:14.053334+0900 | compress | METRIC - time 0.32s
2026-02-04T20:05:14.054475+0900 | compress | METRIC - e

(6/31): Propagating:  60%|██████    | 154/256 [00:56<00:37,  2.71it/s]


KeyboardInterrupt: 

# Model Save

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

# Submission

In [ ]:
zip_name = "baseline_submit"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")